# **Pipeline de Datos y Procesamiento de Audio (DSP)**

| **Integrante 1:** *Bruno Ramos*

## **Bloque 1: Carga, Recorte y Mezcla Digital**

El objetivo de esta fase es preparar los datos de entrada para el modelo NMF simulando un entorno real trabajando con la base de datos ***MUSAN***.

**Justificación del procesamiento:**

1. **Frecuencia de muestreo:** El 100% de los audios en *MUSAN* operan exactamente a **16 kHz**. Esto nos permite omitir el remuestreo y garantiza que la Transformada Corta de Fourier (*STFT*) genere espectrogramas de dimensiones adecuadas para el modelo.

2. **Homogeneización temporal:** Existe una gran variabilidad en la duración de los audios de *MUSAN*, y el ruido suele ser mucho más corto que la voz. Por lo tanto, el proceso incluye recortar las señales a la longitud exacta de la más corta y combinarlas sumando sus valores.

3. **Simulación realista (Atenuación):** Para generar la mezcla final, atenuaremos la pista de ruido un 60% de su amplitud. Esto es algo realista y prepara a la máquina para separar frecuencias superpuestas.

### **Importación de librerías**

In [4]:
import os
import librosa
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt

## **Bloque 1: Función para las mezclas**

In [5]:
def load_and_mix_audio(speech_path, noise_path, sr=16000, noise_attenuation=0.4):
    speech, _ = librosa.load(speech_path, sr=sr)
    noise, _ = librosa.load(noise_path, sr=sr)
    
    # 1. Recortamos la longitud a la del audio más corto
    min_length = min(len(speech), len(noise))
    speech = speech[:min_length]
    noise = noise[:min_length]
    
    # 2. Mezcla digital
    noise_attenuated = noise * noise_attenuation
    mix = speech + noise_attenuated
    
    return speech, noise, mix, sr

speech_file = r"..\data\raw\ejemplo_voz.wav"
noise_file  = r"..\data\raw\ejemplo_ruido.wav"

import os
os.makedirs("../data/processed", exist_ok=True)

speech_signal, noise_signal, mix_signal, sample_rate = load_and_mix_audio(speech_file, noise_file)

sf.write("../data/processed/mix_output.wav", mix_signal, sample_rate)
print("Mezcla guardada en data/processed/")

Mezcla guardada en data/processed/


## **Bloque 2: Transformada de Fourier a Corto Plazo (STFT)**

Para que el modelo NMF pueda separar las fuentes, necesitamos llevar el problema del dominio del tiempo al dominio de la frecuencia.

**Justificación teórica:**
1. **Representación 2D:** La STFT convierte la señal de audio unidimensional $x(t)$ en un espectrograma bidimensional $X \in \mathbb{R}_{\ge0}^{F \times T}$ (conteniendo $F$ bins de frecuencia y $T$ frames de tiempo). Esto nos permite aislar geométricamente las frecuencias de la voz (que son armónicas) de las del ruido (que son estocásticas).

2. **Matriz de Magnitud:** El NMF requiere que todos los datos de entrada sean no-negativos. Por ello, nuestra matriz de entrada será la magnitud de la STFT: $X = |STFT(x)|$.

3. **Preservación de la Fase:** Al tomar solo el valor absoluto perdemos la información temporal fina de la onda. Es obligatorio guardar la fase original ($\angle STFT(x)$) para poder invertir el proceso (iSTFT) al final y recuperar el audio reconstruido sin artefactos graves.

In [6]:
def compute_stft_features(audio_signal, n_fft=2048, hop_length=512):
    stft_matrix = librosa.stft(audio_signal, n_fft=n_fft, hop_length=hop_length)
    
    # Extraemos la Magnitud (Nuestra matriz X para el modelo NMF)
    X_magnitude = np.abs(stft_matrix)
    
    # Extraemos la Fase (Para la reconstrucción posterior)
    X_phase = np.exp(1j * np.angle(stft_matrix))
    
    print(f"Dimensiones de la matriz de magnitud X: {X_magnitude.shape}")
    return X_magnitude, X_phase

# Ejecutamos la función sobre nuestra mezcla obtenida en el bloque anterior
X_mix, phase_mix = compute_stft_features(mix_signal)

Dimensiones de la matriz de magnitud X: (1025, 1267)


## **Bloque 3: Partición Estricta de Datos (*Train/Val/Test*)**

A diferencia de los problemas de clasificación de imágenes (donde mezclar aleatoriamente el dataset es ideal), en el audio los frames de un espectrograma están altamente correlacionados temporalmente. 

Si hiciéramos un *split* aleatorio, un frame $t$ podría quedar en la partición de entrenamiento y el frame $t+1$ en la de prueba. Esto generaría una grave **fuga de datos** (data leakage), haciendo que el set de test no sea un conjunto realmente no visto. Por esto, la partición debe respetar la causalidad temporal usando bloques contiguos: 70% Train, 15% Val y 15% Test.

In [7]:
def contiguous_time_split(X, train_ratio=0.70, val_ratio=0.15):
    total_frames = X.shape[1]
    train_end = int(total_frames * train_ratio)
    val_end = train_end + int(total_frames * val_ratio)
    
    # Cortes en la matriz respetando la línea temporal
    X_train = X[:, :train_end]
    X_val = X[:, train_end:val_end]
    X_test = X[:, val_end:]
    
    print(f"Frames totales: {total_frames}")
    print(f"X_train shape: {X_train.shape} (0 a {train_end})")
    print(f"X_val shape  : {X_val.shape} ({train_end} a {val_end})")
    print(f"X_test shape : {X_test.shape} ({val_end} a {total_frames})")
    
    return X_train, X_val, X_test

# Ejecutamos la partición
X_train, X_val, X_test = contiguous_time_split(X_mix)

Frames totales: 1267
X_train shape: (1025, 886) (0 a 886)
X_val shape  : (1025, 190) (886 a 1076)
X_test shape : (1025, 191) (1076 a 1267)


## **Bloque 4: Reconstrucción del Audio (iSTFT)**

Una vez que el modelo NMF logra separar las fuentes en el dominio de la frecuencia, necesitamos devolver esas señales al dominio del tiempo para poder escucharlas y evaluar el resultado de forma auditiva.

1. **El problema de la magnitud:** El modelo NMF solo trabaja con magnitudes reales positivas ($|STFT(x)|$). Esto significa que al separar la voz del ruido, las matrices resultantes no contienen información de fase, la cual dicta cómo se alinean las ondas en el tiempo.

2. **Re-inyección de la Fase Original:** Para reconstruir el audio $x_{src}$, utilizamos la Transformada Inversa de Fourier a Corto Plazo (iSTFT). El paso crítico aquí es tomar el espectrograma aislado por el NMF ($\hat{X}_{src}$) y multiplicarlo elemento a elemento por la fase compleja original del audio mezclado ($e^{j\angle STFT(x)}$).
 
3. **Resultado:** Aunque usar la fase de la mezcla original no es perfecto (ya que contiene fase de ambas fuentes), es el método estándar más eficiente y produce resultados auditivos muy convincentes sin tener que recurrir a algoritmos iterativos complejos de estimación de fase.

In [8]:
def reconstruct_audio(X_magnitude_separated, original_phase, hop_length=512):
    # Recombinamos la magnitud separada con la fase compleja original
    complex_stft = X_magnitude_separated * original_phase
    
    # Aplicamos la iSTFT para volver a una dimensión
    reconstructed_signal = librosa.istft(complex_stft, hop_length=hop_length)
    
    return reconstructed_signal

audio_reconstruido = reconstruct_audio(X_mix, phase_mix)
sf.write("../data/processed/reconstruccion.wav", audio_reconstruido, 16000)
print("Audio guardado")

Audio guardado


In [9]:
import os
os.makedirs("../data/processed", exist_ok=True)
np.save("../data/processed/X_train.npy", X_train)
np.save("../data/processed/X_val.npy",   X_val)
np.save("../data/processed/X_test.npy",  X_test)
print("Matrices exportadas correctamente.")

Matrices exportadas correctamente.
